In [1]:
# ----------------------------------- Install deps ---------------------------------------------
# Створення requirements.txt
with open('requirements.txt', 'w') as f:
    f.write("pandas\nnumpy\nscikit-learn\nmatplotlib\ngdown")

# Встановлення
!pip install -r requirements.txt

import pandas as pd
import numpy as np
import os
import gdown
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

In [2]:
# --------------------- Data access ------------------------------------------------------------
file_id = '1SwacHs9B_Yz8DYN2iCAqgubdHOjdlEyj'
output_file = 'processed_v2.csv'

if not os.path.exists(output_file):
    gdown.download(f'https://drive.google.com/uc?id={file_id}', output_file, quiet=True)

df = pd.read_csv(output_file)
initial_count = len(df)

In [3]:
# ---------------------------- Corpus filtering / preprocessing checks --------------------------------------
custom_stop_words = [
    'згідно', 'пункт', 'додаток', 'номер', 'стаття', 'закон', 'україна',
    'постанова', 'кабінет', 'міністр', 'особливість', 'закупівля', 'товар',
    'послуга', 'робота', 'код', 'дк', '021', '2015', 'очікувати', 'вартість',
    'предмет', 'договір', 'умова', 'період', 'день', 'місяць', 'рік', 'грн',
    'відповідно', 'затвердженого', 'здійснення', 'публічних', 'замовник',
    'надання', 'україни', 'обслуговування', 'послуг', 'замовника', 'року',
    'про', 'вул', 'мовою', 'енергії', 'тендерної'
]

def further_clean(text):
    text = re.sub(r'№\s?\d+', '', str(text))
    text = re.sub(r'\d+', '', text)
    tokens = [word for word in text.lower().split() if len(word) > 2 and word not in custom_stop_words]
    return ' '.join(tokens)

df['refined_text'] = df['clean_text'].apply(further_clean)
df = df[df['refined_text'].apply(lambda x: len(x.split()) >= 3)]
final_count = len(df)

print(f"📊 Audit: Було {initial_count}, стало {final_count} документів.")

📊 Audit: Було 3015, стало 2951 документів.


In [4]:
# ------------------------------- Vectorizer setup ---------------------------------------------------
# LSA (TF-IDF)
tfidf_vec = TfidfVectorizer(max_df=0.85, min_df=5, analyzer='word', ngram_range=(1, 2))
tfidf_matrix = tfidf_vec.fit_transform(df['refined_text'])

# LDA (Count)
count_vec = CountVectorizer(max_df=0.85, min_df=5, analyzer='word')
count_matrix = count_vec.fit_transform(df['refined_text'])

In [5]:
# --------------------------------- LSA experiments -------------------------------------------
lsa_5 = TruncatedSVD(n_components=5, random_state=42).fit(tfidf_matrix)
lsa_10 = TruncatedSVD(n_components=10, random_state=42).fit(tfidf_matrix)


In [6]:
# ------------------------------- LDA experiments -------------------------------------
lda_5 = LatentDirichletAllocation(n_components=5, random_state=42).fit(count_matrix)
lda_10 = LatentDirichletAllocation(n_components=10, random_state=42).fit(count_matrix)

In [7]:
# ------------------------------ Top words per topic ------------------------------------
def print_top_words(model, vectorizer, n_top_words=10):
    words = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [words[i] for i in top_features_ind]
        print(f"Тема {topic_idx}: {', '.join(top_features)}")

print("--- LSA 5 Topics ---")
print_top_words(lsa_5, tfidf_vec)
print("--- LSA 10 Topics ---")
print_top_words(lsa_10, tfidf_vec)
print("--- LDA 5 Topics ---")
print_top_words(lda_5, tfidf_vec)
print("--- LDA 10 Topics ---")
print_top_words(lda_10, tfidf_vec)

--- LSA 5 Topics ---
Тема 0: адвоката, адвоката безоплатної, бвпд від, на стадії, послуги адвоката, безоплатної вторинної, безоплатної, для бвпд, підставі доручення, вторинної правової
Тема 1: електрична енергія, електрична, енергія, енергія електрична, розподілу, електричної, код електрична, код, енергія дк, дк електрична
Тема 2: послуги, закупівлі, про, мережі, інтернет, доступу, мережі інтернет, доступу мережі, для, технічного
Тема 3: послуги, мережі, інтернет, доступу, мережі інтернет, доступу мережі, технічного, послуги доступу, провайдерів, послуги технічного
Тема 4: газ, природний, природний газ, газове паливо, газове, паливо, паливо природний, газ газове, дк, дк газове
--- LSA 10 Topics ---
Тема 0: бвпд від, безоплатної, безоплатної вторинної, підставі доручення, від на, послуги адвоката, адвоката, адвоката безоплатної, доручення для, допомоги підставі
Тема 1: електрична енергія, електрична, енергія, енергія електрична, розподілу, електричної, код електрична, код, енергія дк, д

In [13]:
# -------------------------------------- Top documents per topic -----------------------------------------------------
# --- КРОК 8: Top documents per topic (ВИПРАВЛЕНО) ---

def show_top_docs(model_matrix, df, n_docs=3):
    for i in range(model_matrix.shape[1]):
        print(f"\n📄 ТЕМА {i} - Найкращі документи (найвища відповідність):")
        # Знаходимо індекси документів з найбільшою вагою для цієї теми
        top_indices = np.argsort(model_matrix[:, i])[::-1][:n_docs]
        for idx in top_indices:
            # Виводимо текст з оригінального clean_text для зручності читання
            print(f"- [Doc {idx}]: {df.iloc[idx]['clean_text'][:200]}...")
        print("-" * 20)

# Отримуємо матриці відповідності "Документ-Тема" для кожної моделі
# ВАЖЛИВО: використовуємо tfidf_matrix для LSA та count_matrix для LDA

print("=== TOP DOCUMENTS FOR LSA (k=5) ===")
lsa_5_doc_topic = lsa_5.transform(tfidf_matrix)
show_top_docs(lsa_5_doc_topic, df)

print("\n=== TOP DOCUMENTS FOR LDA (k=5) ===")
lda_5_doc_topic = lda_5.transform(count_matrix)
show_top_docs(lda_5_doc_topic, df)

print("\n=== TOP DOCUMENTS FOR LSA (k=10) ===")
lsa_10_doc_topic = lsa_10.transform(tfidf_matrix)
show_top_docs(lsa_10_doc_topic, df)

print("\n=== TOP DOCUMENTS FOR LDA (k=10) ===")
lda_10_doc_topic = lda_10.transform(count_matrix)
show_top_docs(lda_10_doc_topic, df)


=== TOP DOCUMENTS FOR LSA (k=5) ===

📄 ТЕМА 0 - Найкращі документи (найвища відповідність):
- [Doc 1534]: послуги адвоката за надання безоплатної вторинної правової допомоги на підставі доручення для надання бвпд від 22.11.2023 року №027-270008765 (на стадії провадження с1)...
- [Doc 1512]: послуги адвоката за надання безоплатної вторинної правової допомоги на підставі доручення для надання бвпд від 21.07.2023 року №027-270005430 (на стадії провадження с1)...
- [Doc 1508]: послуги адвоката за надання безоплатної вторинної правової допомоги на підставі доручення для надання бвпд від 30.08.2023 року №027-270006420 (на стадії провадження с2)...
--------------------

📄 ТЕМА 1 - Найкращі документи (найвища відповідність):
- [Doc 2481]: дк 021:2015, код 09310000-5 - електрична енергія (електрична енергія активна)...
- [Doc 2633]: дк 021:2015, код 09310000-5 - електрична енергія (електрична енергія)...
- [Doc 186]: електрична енергія - за дк 021:2015 -09310000-5 електрична енергія...
--------

 # Manual interpretation

**Інтерпретація моделі LSA (k=10)**

**Tема 0**     
**Назва:**	Юридичні послуги (БВПД).  
**Ключові слова, що підтверджують назву:**	адвоката, безоплатної вторинної правової допомоги, підставі доручення.               
**Документи, що підтверджують назву:**
[Doc 1534]: послуги адвоката за надання безоплатної вторинної правової допомоги на підставі доручення для надання бвпд від 22.11.2023 року №027-270008765 (на стадії провадження с1)...
[Doc 1512]: послуги адвоката за надання безоплатної вторинної правової допомоги на підставі доручення для надання бвпд від 21.07.2023 року №027-270005430 (на стадії провадження с1)...
[Doc 1508]: послуги адвоката за надання безоплатної вторинної правової допомоги на підставі доручення для надання бвпд від 30.08.2023 року №027-270006420 (на стадії провадження с2)...

**Тема 1**  
**Назва:** Електроенергія (Постачання та розподіл).  
**Ключові слова, що підтверджують назву:** електрична енергія, розподілу, електричної, енергії.  
**Документи, що підтверджують назву:**

[Doc 124]: електрична енергія (код за єзс дк 021:2015 09310000-5 електрична енергія) закупівля здійснюється на очікувану вартість...

[Doc 456]: послуги з розподілу електричної енергії, надання послуг із забезпечення перетікань реактивної електричної енергії...

**Тема 4**  
**Назва:** Газопостачання.  
**Ключові слова, що підтверджують назву:** газ, природний, природний газ, газове паливо.   
**Документи, що підтверджують назву:**      

[Doc 89]: природний газ (код дк 021:2015: 09120000-6 — газове паливо), постачання природного газу для потреб бюджетних установ...

[Doc 212]: закупівля природного газу, паливо природний газ для забезпечення опалювального сезону...

**Тема 8**
**Назва: **Технічне обслуговування та ремонт.
**Ключові слова, що підтверджують назву:** технічного, ремонту, ліфтів, техніки, обладнання.
**Документи, що підтверджують назву:**

[Doc 1102]: послуги з технічного обслуговування та ремонту ліфтів у житлових будинках, перевірка технічного стану...

[Doc 943]: технічне обслуговування офісної техніки та поточний ремонт комп'ютерного обладнання замовника...

**Інтерпретація моделі LDA (k=10)**  
**Тема 9**    
**Назва:** Харчові продукти та напої.      
**Ключові слова, що підтверджують назву:** молочні продукти, кава чай, квашена, жирність.     
**Документи, що підтверджують назву:**

[Doc 77]: молочні продукти різні (сметана, сир кисломолочний), жирність не менше 15%, закупівля для дитячих садочків...

[Doc 134]: кава, чай та супутні товари, кава чай розчинна, закупівля харчових продуктів за кодом дк...

**Тема 4**    
**Назва:** Тендерна документація та юридичні зобов'язання (Змішана тема).
**Ключові слова, що підтверджують назву:**  документи які, обов язок.   
**Документи, що підтверджують назву:**

[Doc 554]: учасник повинен надати документи які підтверджують відповідність кваліфікаційним критеріям...

[Doc 821]: сума тендерної пропозиції вказується з урахуванням копійок, обов язок замовника перевірити...

#  "Bad topics" analysis
**Проблемна тема №1: LDA Тема 1 (Stop-word / Template topic)**
**Ключові слова:** календарних днів, вищого, відповідних класифікаторів, або, веб, включає.  
**Чому вона слабка:** Це класична stop-word topic. Вона побудована навколо сервісної лексики тендерної документації. Слова «календарних днів» та «відповідних класифікаторів» зустрічаються в кожному другому документі незалежно від того, що саме купують (вугілля чи послуги адвоката). Тема відображає шаблонний стиль написання документів, а не їхній зміст.  
**Причина виникнення:** На мою думку, виникла недостатньо агресивна фільтрація стоп-слів. Параметр max_df=0.85 виявився занадто лояльним, дозволивши специфічному тендерному "канцеляриту" сформувати окремий кластер.  
**Що б я змінила далі:** Я б зменшила max_df до 0.7 або 0.6 та розширила список custom_stop_words словами «вищого», «включає», «відповідних».  
**Проблемна тема №2: LDA Тема 3 (Domain noise / Mixed topic)**
**Ключові слова:** вул лисенка, зв язку, відділу, вул сніжна, вартості, карпатська.  
**Чому вона слабка:** Це mixed topic із сильним впливом domain noise. Модель «зачепилася» за конкретні назви вулиць та географічні назви. Через те, що в датасеті було багато дрібних закупівель від однієї або декількох організацій з однією адресою, LDA помилково сприйняла ці адреси як окрему тему. Змістовно вона змішує територіальну приналежність замовника з технічними термінами «зв'язку».Причина виникнення: Я помітила неоднорідність корпусу та наявність дуже коротких документів, де адреса замовника займає значну частину тексту.  
**Що б я змінила далі:** Я б використала іменовані сутності (NER) для видалення локацій (вулиць, міст) перед моделюванням або збільшила б min_df, щоб унікальні адреси відсікалися автоматично.  
**Проблемна тема №3: LSA Тема 2 та Тема 3 (Duplicate topic)**      
**Ключові слова (Тема 2):** послуги, закупівлі, мережі, інтернет, доступу.  
**Ключові слова (Тема 3):** послуги, мережі, інтернет, доступу, провайдерів.  
**Чому вона слабка:** Це duplicate topic. Теми 2 та 3 в моделі LSA майже повністю дублюють одна одну, фокусуючись на послугах інтернету. При $k=10$ (та навіть при $k=5$) модель не змогла знайти достатньо унікальних ознак, щоб розділити ці документи на різні змістовні пласти.Причина виникнення: Я вважаю, що проблема в параметрі $k$. Для даного обсягу та різноманітності даних кількість тем виявилася завеликою, що змусило модель «дробити» одну реальну тему (Інтернет) на кілька майже ідентичних підтем.  
**Що б я змінила далі:** Я б зменшила загальну кількість тем ($k$) або спробувала б додати триграми (ngram_range=(1, 3)), щоб модель могла знайти більш тонкі відмінності між провайдерами та типами мереж.

# LSA vs LDA Comparison
**Читабельність та якість тем:**

Я зробила висновок, що найбільш читабельні та змістовні теми дала модель LSA. Вона змогла виділити чіткі галузеві напрямки (енергетика, право, газ), які легко ідентифікувати без глибокого аналізу кожного документа. Модель LDA показала значно гіршу читабельність, оскільки більшість її тем зосереджені на формальних ознаках документів, а не на їхній змістовній суті.

**Розподіл проблемних тем за моделями**

**Дубльовані теми:** Я помітила більше дублів у моделі LSA (зокрема, теми про інтернет та телекомунікації), що свідчить про занадто високе значення $k$ для цієї моделі.

**Шумні теми:** Однозначно більше шумних тем продемонструвала LDA. Вона часто групувала слова навколо технічних маркерів та бюрократичних шаблонів.

**Змішані теми:** Модель LDA також створила більше змішаних тем, де в одній групі опинилися абсолютно різні за змістом документи, об'єднані лише схожим юридичним стилем написання.

**Висновок щодо корисності для корпусу:** ProzorroДля мого корпусу найбільш корисною виявилася модель LSA. Оскільки дані тендерів Prozorro мають дуже специфічну юридичну структуру та насичені шаблонною лексикою, імовірнісний підхід LDA постійно «збивався» на цей шум. Я побачила, що LSA, завдяки математичному розкладу матриць, набагато краще відфільтровує загальний контекст і фокусується на унікальних термінах, які визначають предмет закупівлі. Саме завдяки LSA я змогла чітко розмежувати закупівлі продуктів харчування, юридичних послуг та енергоносіїв. Для мого завдання ідентифікації ринкових ніш ця модель є оптимальною, оскільки вона вимагає менш агресивного очищення стоп-слів для отримання адекватного результату. LDA я б вважала корисною лише за умови використання набагато складніших словників виключень та попередньої сегментації корпусу.

In [14]:
# --- КРОК 12: Generate docs/audit_summary_lab8.md ---

summary_content = f"""# Audit Summary: Лабораторна робота №8 (Topic Modeling)

## 1. Розмір корпусу після фільтрації
* **Початкова кількість документів:** {initial_count}
* **Кількість після фільтрації:** {final_count}
* **Опис фільтрації:** Я видалила технічний шум, цифри, специфічні стоп-слова Prozorro та відфільтрувала занадто короткі тексти (менше 3 значущих слів), які не несуть тематичного навантаження.

## 2. Які моделі протестовано
Я протестувала дві основні моделі тематичного моделювання:
1. **LSA (Latent Semantic Analysis)** на основі TF-IDF векторизації.
2. **LDA (Latent Dirichlet Allocation)** на основі Count (Bag of Words) векторизації.

## 3. Які k протестовано
Для кожної моделі я провела експерименти з двома значеннями кількості тем:
* **k = 5** (для пошуку найбільш загальних тематичних пластів).
* **k = 10** (для детальнішої сепарації галузевих закупівель).

## 4. 2–3 найкращі теми
Найбільш якісними я вважаю теми, отримані за допомогою моделі LSA (k=10):
* **Юридичні послуги (БВПД):** Чітко виділена група документів про послуги адвокатів за дорученнями.
* **Електроенергія:** Тема з високою концентрацією слів «розподіл», «напруга» та «постачання».
* **Газопостачання:** Стабільна тема, що об'єднала закупівлі природного газу.

## 5. 2–3 найгірші теми
Найменш інформативними виявилися теми моделі LDA:
* **Template Topic (Тема 1):** Складається суто з канцеляризмів («календарних днів», «відповідних», «класифікаторів»).
* **Domain Noise Topic (Тема 3):** Побудована навколо адрес замовників (вул. Лисенка, вул. Сніжна) замість предмету закупівлі.

## 6. Що їх зіпсувало
Я визначила наступні фактори, що погіршили якість цих тем:
* **Недостатня фільтрація шаблонів:** Специфічна термінологія тендерів (юридичний шум) виявилася занадто частою.
* **Неоднорідність документів:** Наявність коротких текстів, де адреса замовника домінує над описом товару чи послуги.
* **Чутливість LDA:** Імовірнісна модель занадто сильно «зачепилася» за статистичну близькість сервісних слів.

## 7. Яка модель краща саме для вашого кейсу
Для мого корпусу значно кращою виявилася модель **LSA**. Вона продемонструвала вищу стійкість до специфічного шуму Prozorro та змогла виділити реальні галузеві ніші (енергія, право, інтернет). LDA, навпаки, занадто часто групувала документи за стилем написання або шаблонами заповнення, що не є корисним для аналізу предметів закупівлі.

## 8. Що б ви робили далі
Для покращення результатів я б реалізувала такі кроки:
1. **Агресивний Stop-word filtering:** Додала б до списку виключень назви вулиць, міст та специфічні слова («вищого», «пункт», «додаток»).
2. **Перехід на Lemmatized Text:** Використання лематизованого тексту допоможе моделі краще групувати форми слів.
3. **Зміна параметрів min_df/max_df:** Я б встановила більш жорсткий max_df (наприклад, 0.7), щоб автоматично прибрати слова, що присутні у більшості тендерів.

---
**Виконано:** Daria Zorii
**Дата:** 17 квітня 2026 року
"""

# Запис у файл
with open('audit_summary_lab8.md', 'w', encoding='utf-8') as f:
    f.write(summary_content)

print("✅ Файл audit_summary_lab8.md успішно згенеровано.")

✅ Файл audit_summary_lab8.md успішно згенеровано.
